# CRAFT Stage 3: OpenAI Embedding Reranking

Takes Stage 2 dense-reranked candidates (top-100 per query) and reranks them using OpenAI text embeddings over **mini-tables** (top-5 most relevant rows per table). Outputs top-50 candidates.

**Two modes:**
- `USE_PRECOMPUTED = True` (default): load already-computed rankings from disk — fast, no API calls needed.
- `USE_PRECOMPUTED = False`: build mini-tables, embed with OpenAI API, and rerank from scratch.

**Datasets:**
- `nq`: OpenAI `text-embedding-3-small` or `text-embedding-3-large` (set `EMBED_MODEL`)
- `ottqa`: precomputed rankings (Gemini embeddings); OpenAI used when computing from scratch

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['HF_HOME'] = '/mnt/data2/asing725_2/hf_cache'
# Set HF_TOKEN in your environment before running: export HF_TOKEN=hf_...

CACHE_DIR = '/mnt/data2/asing725_2/hf_cache'

# ── API Keys (replace before running compute mode) ─────────────────────────────
OPENAI_API_KEY = 'your-openai-api-key-here'
GEMINI_API_KEY = 'your-gemini-api-key-here'   # only needed for OTT-QA compute from scratch

In [2]:
import sys
import json
import time
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
from tqdm.notebook import tqdm

# Resolve repo root the same way stage1/stage2 do
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'utils').exists():
    if (REPO_ROOT / 'CRAFT' / 'utils').exists():
        REPO_ROOT = REPO_ROOT / 'CRAFT'
    else:
        REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from utils.io_utils import load_pickle, save_pickle, read_jsonl, write_jsonl
from utils.eval_metric import evaluate_recall

DATA_DIR    = REPO_ROOT / 'datasets'
RESULTS_DIR = REPO_ROOT / 'results' / 'stage3'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root   :', REPO_ROOT)
print('Results dir :', RESULTS_DIR)

Repo root   : /mnt/data1/asing725/ACL/CRAFT
Results dir : /mnt/data1/asing725/ACL/CRAFT/results/stage3


## Configuration

Change `DATASET`, `EMBED_MODEL`, and `USE_PRECOMPUTED` here — everything else is derived automatically.

In [3]:
# ── Main switches ──────────────────────────────────────────────────────────────
DATASET         = 'nq'     # 'nq' or 'ottqa'
EMBED_MODEL     = 'large'  # 'small' or 'large'  (ignored for OTT-QA precomputed — those use Gemini)
USE_PRECOMPUTED = True     # True → load saved rankings; False → call OpenAI API

# ── Pipeline constants ─────────────────────────────────────────────────────────
TOP_K_ROWS   = 5    # rows per table used to build each mini-table
STAGE3_TOP_K = 50   # tables kept per query (final output for next stage)
API_BATCH    = 100  # texts per OpenAI embeddings API call
API_SLEEP    = 0.5  # seconds to sleep between API batches (rate-limit buffer)

RECALL_KS = [1, 5, 10, 20, 50]  # evaluation cutoffs for stage3

# ── OpenAI model ID ───────────────────────────────────────────────────────────
OPENAI_MODEL_ID = 'text-embedding-3-small' if EMBED_MODEL == 'small' else 'text-embedding-3-large'

# ── Dataset-specific paths ─────────────────────────────────────────────────────
if DATASET == 'nq':
    STAGE2_PATH      = REPO_ROOT / 'results' / 'stage2' / 'nq_stage2.jsonl'
    QUESTIONS_PATH   = None
    ROW_RERANK_PATH  = DATA_DIR / 'corpus_2nd_stage_row_rerank_with_ST_original_q.pkl'
    ROW_TABLES_PATH  = DATA_DIR / 'nq_row_tables_INDIV_Table.json'
    # Precomputed: small → nq_stage3_small.jsonl, large → nq_stage3_large.jsonl
    PRECOMPUTED_PATH = DATA_DIR / f'nq_stage3_{EMBED_MODEL}.jsonl'
    OUTPUT_PATH      = RESULTS_DIR / f'nq_stage3_{EMBED_MODEL}.jsonl'

elif DATASET == 'ottqa':
    # Use stage2 notebook output if it exists, else fall back to precomputed stage2
    _s2_out = REPO_ROOT / 'results' / 'stage2' / 'ottqa_stage2.jsonl'
    STAGE2_PATH      = _s2_out if _s2_out.exists() else DATA_DIR / 'ottqa_stage2_results.jsonl'
    QUESTIONS_PATH   = Path('/mnt/data1/asing725/ottqa/data/OTT_QA_Qeuery_Desc.jsonl')
    ROW_RERANK_PATH  = DATA_DIR / 'ottqa_top_rows.pkl'
    ROW_TABLES_PATH  = DATA_DIR / 'ottqa_row_tables.pkl'
    PRECOMPUTED_PATH = DATA_DIR / 'ottqa_stage3_results.jsonl'   # Gemini embeddings
    OUTPUT_PATH      = RESULTS_DIR / 'ottqa_stage3.jsonl'

else:
    raise ValueError(f'Unknown dataset: {DATASET!r}. Use "nq" or "ottqa".')

print(f'Dataset        : {DATASET}')
print(f'Embed model    : {EMBED_MODEL}  →  {OPENAI_MODEL_ID}')
print(f'Use precomputed: {USE_PRECOMPUTED}')
print(f'Stage2 input   : {STAGE2_PATH}')
print(f'Precomputed    : {PRECOMPUTED_PATH}')
print(f'Output path    : {OUTPUT_PATH}')

Dataset        : nq
Embed model    : large  →  text-embedding-3-large
Use precomputed: True
Stage2 input   : /mnt/data1/asing725/ACL/CRAFT/results/stage2/nq_stage2.jsonl
Precomputed    : /mnt/data1/asing725/ACL/CRAFT/datasets/nq_stage3_large.jsonl
Output path    : /mnt/data1/asing725/ACL/CRAFT/results/stage3/nq_stage3_large.jsonl


## Step 1 — Load Stage 2 Results

Stage 2 provides the top-100 candidate tables per query that Stage 3 will rerank.

Both datasets are normalised to the unified internal format:
```json
{"qid": "...", "question": "...", "gold_table_ids": ["..."], "retrieved": [{"rank": 1, "table_id": "..."}]}
```

**NQ** — `results/stage2/nq_stage2.jsonl` already in this format.  
**OTT-QA** — unified output if stage2.ipynb was run; otherwise the raw precomputed format
`{qid, gold_table_id, ranked_tables: [{table_id, score}]}` is normalised here.

In [4]:
if DATASET == 'nq':
    stage2_results = list(read_jsonl(STAGE2_PATH))

elif DATASET == 'ottqa':
    raw_s2 = list(read_jsonl(STAGE2_PATH))

    if raw_s2 and 'retrieved' in raw_s2[0]:
        # Unified format (from results/stage2/ottqa_stage2.jsonl)
        stage2_results = raw_s2
        print(f'Stage 2 (unified format): {len(stage2_results)} questions')
    else:
        # Raw precomputed: {qid, gold_table_id, ranked_tables}
        # Recover question text from the questions file
        questions_raw   = list(read_jsonl(QUESTIONS_PATH))
        qid_to_question = {it['question_id']: it['question'].strip() for it in questions_raw}
        stage2_results  = []
        for item in raw_s2:
            qid      = item['qid']
            ranked   = item.get('ranked_tables', [])
            retrieved = [
                {'rank': r + 1, 'table_id': t['table_id'], 'score': t.get('score')}
                for r, t in enumerate(ranked)
            ]
            stage2_results.append({
                'qid'           : qid,
                'question'      : qid_to_question.get(qid, ''),
                'gold_table_ids': [item['gold_table_id']],
                'retrieved'     : retrieved,
            })
        print(f'Stage 2 (normalised from raw precomputed): {len(stage2_results)} questions')

stage2_by_qid = {it['qid']: it for it in stage2_results}

print(f'Stage 2 results : {len(stage2_results)} questions')
ex = stage2_results[0]
print(f'Example qid     : {ex["qid"]}')
print(f'Question        : {ex["question"][:70]}')
print(f'Gold table IDs  : {ex["gold_table_ids"]}')
print(f'Candidates      : {len(ex["retrieved"])} tables (stage2 top-k)')

Stage 2 results : 966 questions
Example qid     : dev_6330519627947400943_0
Question        : where does the brazos river start and stop
Gold table IDs  : ['Brazos River_8F7B4BA175AC5E8F']
Candidates      : 100 tables (stage2 top-k)


## Step 2 — Build Row-Data Index

Needed only when `USE_PRECOMPUTED = False` to look up row text when building mini-tables.

- **NQ** — `nq_row_tables_INDIV_Table.json`: `{Table Id, table_row_number, Row Data}`
- **OTT-QA** — `ottqa_row_tables.pkl`: flat list of row dicts; indexed by absolute row index

**Skipped** when `USE_PRECOMPUTED = True`.

In [5]:
row_index = None

if not USE_PRECOMPUTED:
    print('Building row-data index …')

    if DATASET == 'nq':
        with open(ROW_TABLES_PATH, 'r') as f:
            raw_rows = json.load(f)
        # row_index[table_id][row_number] = row_text
        row_index = defaultdict(dict)
        for row in raw_rows:
            row_index[row['Table Id']][row['table_row_number']] = row.get('Row Data', '')
        print(f'  NQ: {len(row_index):,} tables, {len(raw_rows):,} total rows')

    elif DATASET == 'ottqa':
        try:
            row_index = load_pickle(ROW_TABLES_PATH)
            print(f'  OTT-QA: {len(row_index):,} rows loaded')
        except Exception as e:
            print(f'  WARNING: {e}')
            print('  Mini-table construction will fall back to table_id strings.')
            row_index = []
else:
    print('Skipping row index (USE_PRECOMPUTED=True)')

Skipping row index (USE_PRECOMPUTED=True)


## Step 3 — Load Row-Rank Data

Maps each *(question, table)* pair to the top-K most relevant row indices, pre-ranked by the Stage 2 Sentence Transformer.

- **NQ** → `{ qid → { table_id → [table_row_number, …] } }`  (0-indexed within table)
- **OTT-QA** → `{ qid → { table_id → [abs_row_index, …] } }`  (absolute index into row list)

**Skipped** when `USE_PRECOMPUTED = True`.

In [6]:
row_rerank = None

if not USE_PRECOMPUTED:
    row_rerank = load_pickle(ROW_RERANK_PATH)
    print(f'Row-rerank entries: {len(row_rerank):,} questions')
    sample_qid = next(iter(row_rerank))
    for tid, rows in list(row_rerank[sample_qid].items())[:2]:
        print(f'  {tid}: top rows = {rows[:TOP_K_ROWS]}')
else:
    print('Skipping row-rerank load (USE_PRECOMPUTED=True)')

Skipping row-rerank load (USE_PRECOMPUTED=True)


## Step 4 — Mini-Table Builder

For each *(question, table)* pair:
1. Look up the top-`TOP_K_ROWS` row indices from `row_rerank`.
2. Fetch each row's text from `row_index`.
3. Concatenate into a single string — the **mini-table**.

Falls back to the `table_id` string if no row data is found.
Only active when `USE_PRECOMPUTED = False`.

In [7]:
def build_minitable_nq(qid: str, table_id: str) -> str:
    top_row_nums  = row_rerank.get(qid, {}).get(table_id, [])[:TOP_K_ROWS]
    table_row_map = row_index.get(table_id, {})
    texts = [table_row_map[rn].strip() for rn in top_row_nums if table_row_map.get(rn, '').strip()]
    return ' '.join(texts) if texts else table_id.replace('_', ' ')


def build_minitable_ottqa(qid: str, table_id: str) -> str:
    abs_indices = row_rerank.get(qid, {}).get(table_id, [])[:TOP_K_ROWS]
    texts = []
    for idx in abs_indices:
        if idx < len(row_index):
            row  = row_index[idx]
            text = row.get('Row Data', row.get('row_data', '')).strip()
            if text:
                texts.append(text)
    return ' '.join(texts) if texts else table_id.replace('_', ' ')


build_minitable = build_minitable_nq if DATASET == 'nq' else build_minitable_ottqa

if not USE_PRECOMPUTED and row_rerank is not None:
    ex_qid   = stage2_results[0]['qid']
    ex_table = stage2_results[0]['retrieved'][0]['table_id']
    mt = build_minitable(ex_qid, ex_table)
    print('Mini-table preview:')
    print(' ', mt[:300])
else:
    print('Mini-table builder defined (used only when USE_PRECOMPUTED=False)')

Mini-table builder defined (used only when USE_PRECOMPUTED=False)


## Step 5 — Load or Compute Stage 3 Rankings

### 5a. Load pre-computed results (default, `USE_PRECOMPUTED = True`)

All three precomputed files now share the same unified format:
```json
{"qid": "...", "question": "...", "gold_table_ids": ["..."], "gold_rank": 1,
 "retrieved": [{"rank": 1, "table_id": "...", "score": 0.64}]}
```

| Dataset | File | Embedding |
|---------|------|-----------|
| NQ (small) | `nq_stage3_small.jsonl` | OpenAI text-embedding-3-small |
| NQ (large) | `nq_stage3_large.jsonl` | OpenAI text-embedding-3-large (966q, actual-rerank-corpus) |
| OTT-QA | `ottqa_stage3_results.jsonl` | Gemini embedding-001 |

Because the format is unified, the same loading logic applies to both datasets.

### 5b. Compute from scratch (`USE_PRECOMPUTED = False`)

For each question:
1. Build mini-table strings for the top-100 Stage 2 candidates.
2. Call OpenAI embeddings API to embed the question and all mini-tables.
3. Rank by cosine similarity (dot product of L2-normalised vectors).

In [8]:
stage3_results = []

# ══════════════════════════════════════════════════════════════════════════════
# 5a  PRECOMPUTED  (unified format — same loading for NQ and OTT-QA)
# ══════════════════════════════════════════════════════════════════════════════
if USE_PRECOMPUTED:
    print('Loading pre-computed Stage 3 rankings …')
    precomp_list   = list(read_jsonl(PRECOMPUTED_PATH))
    precomp_by_qid = {it['qid']: it for it in precomp_list}
    print(f'  File : {PRECOMPUTED_PATH.name}  ({len(precomp_list)} entries)')

    matched, missing = 0, 0
    for s2_item in stage2_results:
        pc = precomp_by_qid.get(s2_item['qid'], {})
        if not pc:
            missing += 1
        ranked    = pc.get('retrieved', [])[:STAGE3_TOP_K]
        retrieved = [
            {'rank': t.get('rank', r + 1), 'table_id': t['table_id'], 'score': t.get('score')}
            for r, t in enumerate(ranked)
        ]
        stage3_results.append({
            'qid'           : s2_item['qid'],
            'question'      : s2_item['question'],
            'gold_table_ids': s2_item['gold_table_ids'],
            'gold_rank'     : pc.get('gold_rank'),
            'retrieved'     : retrieved,
        })
        matched += 1

    print(f'  Matched {matched} questions ({missing} without precomputed entry → empty retrieved)')
    print(f'Stage 3 assembled: {len(stage3_results)} questions')


# ══════════════════════════════════════════════════════════════════════════════
# 5b  COMPUTE FROM SCRATCH  (OpenAI embeddings)
# ══════════════════════════════════════════════════════════════════════════════
else:
    from openai import OpenAI

    oai_client = OpenAI(api_key=OPENAI_API_KEY)
    print(f'OpenAI model : {OPENAI_MODEL_ID}')
    print(f'API batch    : {API_BATCH} texts per call')

    def openai_embed(texts: list, model: str = OPENAI_MODEL_ID) -> np.ndarray:
        """Embed texts with OpenAI API; returns L2-normalised float32 matrix (N, D)."""
        all_embs = []
        for i in range(0, len(texts), API_BATCH):
            batch = texts[i : i + API_BATCH]
            resp  = oai_client.embeddings.create(input=batch, model=model)
            batch_embs = [np.array(e.embedding, dtype=np.float32) for e in resp.data]
            all_embs.extend(batch_embs)
            if i + API_BATCH < len(texts):
                time.sleep(API_SLEEP)
        mat   = np.array(all_embs)                              # (N, D)
        norms = np.linalg.norm(mat, axis=1, keepdims=True)
        return mat / np.maximum(norms, 1e-10)                   # L2-normalised

    for item in tqdm(stage2_results, desc='Stage 3 reranking'):
        qid        = item['qid']
        question   = item['question']
        candidates = [r['table_id'] for r in item['retrieved']]  # up to 100

        mini_tables = [build_minitable(qid, tid) for tid in candidates]
        valid_pairs = [(tid, mt) for tid, mt in zip(candidates, mini_tables) if mt.strip()]
        if not valid_pairs:
            stage3_results.append({
                'qid': qid, 'question': question,
                'gold_table_ids': item['gold_table_ids'], 'gold_rank': None, 'retrieved': [],
            })
            continue

        valid_ids, valid_texts = zip(*valid_pairs)
        all_vecs = openai_embed([question] + list(valid_texts))
        q_vec    = all_vecs[0]
        mt_vecs  = all_vecs[1:]

        scores     = mt_vecs @ q_vec
        ranked_idx = np.argsort(-scores)
        top_idx    = ranked_idx[:STAGE3_TOP_K]

        gold_ids  = item['gold_table_ids']
        retrieved = [
            {'rank': r + 1, 'table_id': valid_ids[i], 'score': float(scores[i])}
            for r, i in enumerate(top_idx)
        ]
        # compute gold_rank
        ret_ids   = [t['table_id'] for t in retrieved]
        gold_rank = next((ret_ids.index(g) + 1 for g in gold_ids if g in ret_ids), None)

        stage3_results.append({
            'qid'           : qid,
            'question'      : question,
            'gold_table_ids': gold_ids,
            'gold_rank'     : gold_rank,
            'retrieved'     : retrieved,
        })

    print(f'Computed Stage 3 for {len(stage3_results)} questions')

Loading pre-computed Stage 3 rankings …
  File : nq_stage3_large.jsonl  (966 entries)
  Matched 966 questions (41 without precomputed entry → empty retrieved)
Stage 3 assembled: 966 questions


## Step 6 — Save Results

Output format (same for both datasets):
```json
{"qid": "...", "question": "...", "gold_table_ids": ["..."], "retrieved": [{"rank": 1, "table_id": "..."}]}
```

In [9]:
write_jsonl(OUTPUT_PATH, stage3_results)
print(f'Saved {len(stage3_results)} results → {OUTPUT_PATH}')

ex = stage3_results[0]
print('\nExample output:')
print('  qid           :', ex['qid'])
print('  question      :', ex['question'][:70])
print('  gold_table_ids:', ex['gold_table_ids'])
print(f'  retrieved     : {len(ex["retrieved"])} tables (top-{STAGE3_TOP_K})')
print('  top-3 tables  :', [r['table_id'] for r in ex['retrieved'][:3]])

Saved 966 results → /mnt/data1/asing725/ACL/CRAFT/results/stage3/nq_stage3_large.jsonl

Example output:
  qid           : dev_6330519627947400943_0
  question      : where does the brazos river start and stop
  gold_table_ids: ['Brazos River_8F7B4BA175AC5E8F']
  retrieved     : 5 tables (top-50)
  top-3 tables  : ['Brazos River_8F7B4BA175AC5E8F', 'List of longest rivers of the United States (by main stem)_4FED3AEAE3027604', 'Colorado River (Texas)_DCE6AAC8E90FC40A']


## Step 7 — Before / After Recall Comparison

Side-by-side Recall@k for **Stage 2 (Dense Reranking)** vs **Stage 3 (OpenAI Embedding)** across both datasets.

Evaluation is restricted to questions present in **both** Stage 2 and Stage 3 results for a fair comparison.

> **Note on Recall@50:** Stage 3 outputs only the top-`STAGE3_TOP_K` (50) tables, so Recall@50 is the ceiling. The drop vs Stage 2 at high k is expected — Stage 3 trades breadth for precision at the top ranks.

> **Improvement formula:** `(S3 − S2) / S2 × 100`  (relative gain over Stage 2)

In [10]:
import re
import pandas as pd
from IPython.display import display

EVAL_KS = RECALL_KS  # [1, 5, 10, 20, 50]

def to_recall_fmt(items):
    return [{'qid': it['qid'], 'gold_table_ids': it['gold_table_ids'],
             'retrieved': it['retrieved']} for it in items]

def pct_improvement(s2, s3):
    """Relative improvement: (s3 - s2) / s2 * 100."""
    if s2 == 0:
        return 'N/A'
    return f"{(s3 - s2) / s2 * 100:+.1f}%"

def recall_from_ranks(gold_ranks, ks):
    """Compute Recall@k directly from a list of integer gold ranks."""
    n = len(gold_ranks)
    return {k: sum(1 for r in gold_ranks if r <= k) / n for k in ks}


# ── Unified loader using `retrieved` list (for complete files) ────────────────
def load_stage3_file(path, s2_by_qid):
    """Load unified stage3 jsonl; pair with stage2 baseline by qid."""
    s3_list = list(read_jsonl(path))
    s2_aln, s3_aln = [], []
    for it in s3_list:
        s2i = s2_by_qid.get(it['qid'])
        if s2i is None:
            continue
        top = it.get('retrieved', [])[:STAGE3_TOP_K]
        s2_aln.append(s2i)
        s3_aln.append({
            'qid'           : it['qid'],
            'gold_table_ids': it['gold_table_ids'],
            'retrieved'     : [{'rank': t.get('rank', r+1), 'table_id': t['table_id'],
                                'score': t.get('score')} for r, t in enumerate(top)],
        })
    return s2_aln, s3_aln


# ── Gold-rank loader (for NQ large — only 5 retrieved entries stored per query) ─
# The txt source only saved top-5 display entries; Gold Rank is the authoritative rank.
def load_stage3_goldrank(path, s2_by_qid, ks):
    """Compute recall using gold_rank field; avoids incomplete retrieved lists."""
    s3_list = list(read_jsonl(path))
    s2_aln, gold_ranks = [], []
    for it in s3_list:
        s2i = s2_by_qid.get(it['qid'])
        if s2i is None:
            continue
        gr = it.get('gold_rank')
        s2_aln.append(s2i)
        gold_ranks.append(gr if gr is not None else 999999)
    return s2_aln, recall_from_ranks(gold_ranks, ks), len(gold_ranks)


# ── NQ stage2 index ───────────────────────────────────────────────────────────
nq_s2_list   = list(read_jsonl(REPO_ROOT / 'results' / 'stage2' / 'nq_stage2.jsonl'))
nq_s2_by_qid = {it['qid']: it for it in nq_s2_list}

# NQ small — full retrieved list available → use list-based eval
nq_s2_sm, nq_s3_sm = load_stage3_file(DATA_DIR / 'nq_stage3_small.jsonl', nq_s2_by_qid)
nq_s2m_sm = evaluate_recall(to_recall_fmt(nq_s2_sm), EVAL_KS)
nq_s3m_sm = evaluate_recall(to_recall_fmt(nq_s3_sm), EVAL_KS)
print(f'NQ small  : {len(nq_s3_sm)} questions (list-based eval)')

# NQ large — only top-5 stored for most queries → use gold_rank field for recall
nq_s2_lg, nq_s3m_lg, n_lg = load_stage3_goldrank(
    DATA_DIR / 'nq_stage3_large.jsonl', nq_s2_by_qid, EVAL_KS)
nq_s2m_lg = evaluate_recall(to_recall_fmt(nq_s2_lg), EVAL_KS)
print(f'NQ large  : {n_lg} questions (gold_rank-based eval — txt log stores top-5 only)')

# ── OTT-QA stage2 index ───────────────────────────────────────────────────────
ott_s2_raw    = list(read_jsonl(DATA_DIR / 'ottqa_stage2_results.jsonl'))
ott_s2_by_qid = {}
for it in ott_s2_raw:
    top = it.get('ranked_tables', [])[:STAGE3_TOP_K]
    ott_s2_by_qid[it['qid']] = {
        'qid'           : it['qid'],
        'gold_table_ids': [it['gold_table_id']],
        'retrieved'     : [{'rank': r+1, 'table_id': t['table_id']} for r, t in enumerate(top)],
    }

# OTT-QA — full retrieved list available → list-based eval
ott_s2_aln, ott_s3_aln = load_stage3_file(DATA_DIR / 'ottqa_stage3_results.jsonl', ott_s2_by_qid)
ott_s2m = evaluate_recall(to_recall_fmt(ott_s2_aln), EVAL_KS)
ott_s3m = evaluate_recall(to_recall_fmt(ott_s3_aln), EVAL_KS)
print(f'OTT-QA    : {len(ott_s3_aln)} questions (list-based eval)')

# ── Build comparison table ────────────────────────────────────────────────────
rows = []
for k in EVAL_KS:
    rows.append({
        'k'                         : k,
        # NQ small
        'NQ Stage2 (baseline)'      : f"{nq_s2m_sm[k]:.4f}",
        'NQ S3 small'               : f"{nq_s3m_sm[k]:.4f}",
        'NQ Small Impr.'            : pct_improvement(nq_s2m_sm[k], nq_s3m_sm[k]),
        # NQ large (gold_rank-based)
        'NQ S3 large (n=966)'       : f"{nq_s3m_lg[k]:.4f}",
        'NQ Large Impr.'            : pct_improvement(nq_s2m_lg[k], nq_s3m_lg[k]),
        # OTT-QA
        'OTT-QA Stage2 (baseline)'  : f"{ott_s2m[k]:.4f}",
        'OTT-QA S3 (Gemini)'        : f"{ott_s3m[k]:.4f}",
        'OTT-QA Impr.'              : pct_improvement(ott_s2m[k], ott_s3m[k]),
    })

df = pd.DataFrame(rows).set_index('k')

print(f'\n{"=" * 90}')
print(f'  Stage 2 → Stage 3 Recall@k')
print(f'  NQ small    : text-embedding-3-small  (n={len(nq_s3_sm)})  [list eval]')
print(f'  NQ large    : text-embedding-3-large  (n={n_lg})   [gold_rank eval — txt log, actual-rerank-corpus]')
print(f'  OTT-QA      : Gemini embedding-001     (n={len(ott_s3_aln)})  [list eval]')
print(f'  Mode        : {"precomputed" if USE_PRECOMPUTED else "computed from scratch"}')
print(f'  Stage3 top-k: {STAGE3_TOP_K}')
print(f'  Improvement = relative gain over Stage 2  [ (S3−S2)/S2 × 100 ]')
print(f'{"=" * 90}')
display(df)

# ── Persist summary CSV ───────────────────────────────────────────────────────
summary_path = RESULTS_DIR / 'recall_summary.csv'
new_rows = []
for dataset, embed, stage, metrics, n in [
    ('NQ-Tables', 'text-embedding-3-small',      'stage2', nq_s2m_sm, len(nq_s2_sm)),
    ('NQ-Tables', 'text-embedding-3-small',      'stage3', nq_s3m_sm, len(nq_s3_sm)),
    ('NQ-Tables', 'text-embedding-3-large',      'stage2', nq_s2m_lg, len(nq_s2_lg)),
    ('NQ-Tables', 'text-embedding-3-large',      'stage3', nq_s3m_lg, n_lg),
    ('OTT-QA',    'gemini-embedding-001',         'stage2', ott_s2m,   len(ott_s2_aln)),
    ('OTT-QA',    'gemini-embedding-001',         'stage3', ott_s3m,   len(ott_s3_aln)),
]:
    row = {
        'dataset'     : dataset,
        'stage'       : stage,
        'embed_model' : embed,
        'mode'        : 'precomputed' if USE_PRECOMPUTED else 'computed',
        'num_queries' : n,
    }
    for k in EVAL_KS:
        row[f'recall@{k}'] = metrics[k]
    new_rows.append(row)

new_df = pd.DataFrame(new_rows)
if summary_path.exists():
    combined = pd.concat([pd.read_csv(summary_path), new_df], ignore_index=True)
else:
    combined = new_df
combined.to_csv(summary_path, index=False)
print(f'\nSummary saved → {summary_path}')


NQ small  : 919 questions (list-based eval)
NQ large  : 966 questions (gold_rank-based eval — txt log stores top-5 only)
OTT-QA    : 2214 questions (list-based eval)

  Stage 2 → Stage 3 Recall@k
  NQ small    : text-embedding-3-small  (n=919)  [list eval]
  NQ large    : text-embedding-3-large  (n=966)   [gold_rank eval — txt log, actual-rerank-corpus]
  OTT-QA      : Gemini embedding-001     (n=2214)  [list eval]
  Mode        : precomputed
  Stage3 top-k: 50
  Improvement = relative gain over Stage 2  [ (S3−S2)/S2 × 100 ]


,NQ Stage2 (baseline),NQ S3 small,NQ Small Impr.,NQ S3 large (n=966),NQ Large Impr.,OTT-QA Stage2 (baseline),OTT-QA S3 (Gemini),OTT-QA Impr.
k,,,,,,,,
1,0.3645,0.4113,+12.8%,0.4990,+39.3%,0.4734,0.5556,+17.4%
5,0.6997,0.7737,+10.6%,0.8075,+16.6%,0.7407,0.8473,+14.4%
10,0.7900,0.8705,+10.2%,0.8716,+10.8%,0.8189,0.8988,+9.8%
20,0.8683,0.9293,+7.0%,0.9441,+8.8%,0.8735,0.9332,+6.8%
50,0.9293,0.9706,+4.4%,0.9710,+4.6%,0.9182,0.9607,+4.6%



Summary saved → /mnt/data1/asing725/ACL/CRAFT/results/stage3/recall_summary.csv
